# GameForge AI: Action & Pose Diffusion Training (50,000 Verified Pairs)
### Module 4: 50,000 Paired Samples — Dual GPU Accelerated (Kaggle T4 x2)

---
### Ground Truth Guarantee:
- **Full Scale Corpus**: 3,125 distinct character spritesheets $\times$ 16 active action frames = **50,000 verified intra-sheet training pairs**.
- **Reference Sprite**: Frame (Col 0, Row 0) of each sheet (Canonical Idle Stance).
- **Target Action**: Active combat frames from the **EXACT SAME sheet**.
- **Action ID Conditioning**: 0: Spellcast, 1: Thrust, 2: Walk, 3: Slash.
- **Dual GPU Optimization**: `BATCH_SIZE = 128` (64 per GPU), Mixed Precision (AMP), Checkpointing after each epoch.

In [ ]:
# Cell 1: Environment Setup & Dual GPU Detection
import os
import sys
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"⚡ Compute Target: {DEVICE} ({NUM_GPUS} GPUs Available)")
if NUM_GPUS > 0:
    for i in range(NUM_GPUS):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./diffusion_workspace")
WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = WORK / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK)

In [ ]:
# Cell 2: Stream LPC Action Dataset from Hugging Face
!pip install -q datasets huggingface_hub diffusers torchvision
from datasets import load_dataset

ACTION_DATASET_ID = "carlosuperb/lpc-action-pixel-art-diffusion"
print(f"Loading verified action dataset: {ACTION_DATASET_ID}...")
action_ds = load_dataset(ACTION_DATASET_ID, split="train")
print(f"Loaded {len(action_ds):,} character action sheets!")

In [ ]:
# Cell 3: Intra-Sheet Verified Paired Construction (50,000 Pairs)
GRID_COLS = 9
GRID_ROWS = 16
IMAGE_SIZE = 64
MAX_SHEETS = 3125 # 3,125 sheets * 16 action poses = 50,000 verified pairs

# 4 Primary Combat Actions
# Rows 0-3: Spellcast | Rows 4-7: Thrust | Rows 8-11: Walk | Rows 12-15: Slash
ACTION_NAMES = ["Spellcast", "Thrust", "Walk", "Slash"]

pairs = []
print(f"Extracting 50,000 verified intra-sheet pairs from {MAX_SHEETS} characters...")

limit = min(MAX_SHEETS, len(action_ds))
for idx in tqdm(range(limit), desc="Slicing 50,000 Verified Pairs"):
    try:
        sheet = action_ds[idx]["image"].convert("RGBA")
        cw = sheet.width // GRID_COLS
        ch = sheet.height // GRID_ROWS
        
        # Reference: Canonical Idle frame (Row 0, Col 0) of THIS specific character
        idle_crop = sheet.crop((0, 0, cw, ch)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)
        ref_t = torch.from_numpy(np.asarray(idle_crop, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1)[:3]
        
        # Extract 16 distinct active action frames from the SAME character
        for r in range(min(16, GRID_ROWS)):
            action_type = r // 4 # 0: Cast, 1: Thrust, 2: Walk, 3: Slash
            c = 2 # Active stride/strike frame column
            box = (c * cw, r * ch, (c + 1) * cw, (r + 1) * ch)
            crop = sheet.crop(box)
            if np.asarray(crop.getchannel("A")).sum() < 40:
                continue
            crop_64 = crop.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)
            tgt_t = torch.from_numpy(np.asarray(crop_64, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1)[:3]
            pairs.append((ref_t, tgt_t, action_type))
    except Exception:
        continue

print(f"✅ Successfully paired {len(pairs):,} verified intra-character training pairs!")

class VerifiedActionDataset(Dataset):
    def __init__(self, sample_list):
        self.samples = sample_list

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ref, tgt, act_type = self.samples[idx]
        return {
            "reference": ref, 
            "target": tgt, 
            "action_id": torch.tensor(act_type, dtype=torch.long)
        }

dataset = VerifiedActionDataset(pairs)
PER_GPU_BATCH = 64
BATCH_SIZE = PER_GPU_BATCH * max(1, NUM_GPUS)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, drop_last=True)
print(f"DataLoader: Batch Size = {BATCH_SIZE} | Total Batches per Epoch = {len(train_loader)}")

In [ ]:
# Cell 4: Action-Conditioned Diffusion U-Net
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = 10000 ** (torch.arange(half_dim, device=device).float() / (half_dim - 1))
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ActionConditionedDiffusionUNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=3, time_dim=128, num_actions=4, base=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU()
        )
        # Action Condition Embedding (0: Cast, 1: Thrust, 2: Walk, 3: Slash)
        self.action_embed = nn.Embedding(num_actions, time_dim)
        self.cond_combine = nn.Linear(time_dim * 2, time_dim)
        
        self.in_conv = nn.Conv2d(in_channels, base, 3, padding=1)
        self.down1 = nn.Conv2d(base, base * 2, 4, 2, 1)
        self.time_emb1 = nn.Linear(time_dim, base * 2)
        
        self.down2 = nn.Conv2d(base * 2, base * 4, 4, 2, 1)
        self.time_emb2 = nn.Linear(time_dim, base * 4)
        
        self.mid_conv = nn.Conv2d(base * 4, base * 4, 3, padding=1)
        
        self.up1 = nn.ConvTranspose2d(base * 4, base * 2, 4, 2, 1)
        self.up2 = nn.ConvTranspose2d(base * 2, base, 4, 2, 1)
        self.out_conv = nn.Conv2d(base, out_channels, 3, padding=1)
        
    def forward(self, noisy_target, timestep, reference, action_id):
        t_emb = self.time_mlp(timestep)
        a_emb = self.action_embed(action_id)
        cond_emb = F.silu(self.cond_combine(torch.cat([t_emb, a_emb], dim=-1)))
        
        x = torch.cat([noisy_target, reference], dim=1) # 6 channels
        
        h0 = F.silu(self.in_conv(x))
        h1 = F.silu(self.down1(h0) + self.time_emb1(cond_emb)[:, :, None, None])
        h2 = F.silu(self.down2(h1) + self.time_emb2(cond_emb)[:, :, None, None])
        
        m = F.silu(self.mid_conv(h2))
        
        u1 = F.silu(self.up1(m) + h1)
        u2 = F.silu(self.up2(u1) + h0)
        return self.out_conv(u2)

TIMESTEPS = 1000
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(DEVICE)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0).to(DEVICE)

raw_model = ActionConditionedDiffusionUNet(time_dim=128, base=64, num_actions=4).to(DEVICE)
print(f"Action-Conditioned UNet built: {sum(p.numel() for p in raw_model.parameters()):,} parameters.")

if NUM_GPUS > 1:
    print(f"⚡ Activating torch.nn.DataParallel across {NUM_GPUS} GPUs!")
    model = nn.DataParallel(raw_model)
else:
    model = raw_model

In [ ]:
# Cell 5: Strict 50-Epoch Dual GPU Training Loop
EPOCHS = 50
LR = 2e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

start_epoch = 1
best_loss = float("inf")
history = []

resume_ckpt = CHECKPOINT_DIR / "last_action_diffusion_50k_50ep.pt"
if resume_ckpt.exists():
    print(f"🔄 Resuming from {resume_ckpt}...")
    ckpt = torch.load(resume_ckpt, map_location=DEVICE)
    state_dict = ckpt["model_state_dict"]
    if NUM_GPUS > 1 and not list(state_dict.keys())[0].startswith("module."):
        state_dict = {"module." + k: v for k, v in state_dict.items()}
    elif NUM_GPUS <= 1 and list(state_dict.keys())[0].startswith("module."):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_loss = ckpt.get("loss", float("inf"))
    if (WORK / "action_diffusion_50k_50ep_history.csv").exists():
        history = pd.read_csv(WORK / "action_diffusion_50k_50ep_history.csv").to_dict("records")
    print(f"✅ Resumed from Epoch {start_epoch}! Best loss: {best_loss:.5f}")
else:
    print("✨ Starting brand-new 50K Action-Conditioned Diffusion training.")

print(f"🚀 Training 50 Strict Epochs on {DEVICE} (Batch Size = {BATCH_SIZE})...")
start_time = time.time()

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    model.train()
    epoch_loss_sum, epoch_batches = 0.0, 0
    
    for batch in train_loader:
        ref = batch["reference"].to(DEVICE, non_blocking=True)
        x0 = batch["target"].to(DEVICE, non_blocking=True)
        act_id = batch["action_id"].to(DEVICE, non_blocking=True)
        b = x0.size(0)
        
        t = torch.randint(0, TIMESTEPS, (b,), device=DEVICE).long()
        noise = torch.randn_like(x0)
        
        a_bar = alphas_cumprod[t][:, None, None, None]
        xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise
        
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            pred_noise = model(xt, t, ref, act_id)
            loss = F.mse_loss(pred_noise, noise)
            
        if not torch.isfinite(loss):
            continue
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss_sum += loss.item() * b
        epoch_batches += b
        
    train_loss = epoch_loss_sum / max(1, epoch_batches)
    history.append({"epoch": epoch, "diffusion_loss": train_loss, "lr": LR})
    ep_dur = time.time() - t0
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss: {train_loss:.5f} - Time: {ep_dur:.1f}s")
    
    payload = {
        "epoch": epoch,
        "loss": train_loss,
        "model_state_dict": model.module.state_dict() if NUM_GPUS > 1 else model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict()
    }
    torch.save(payload, CHECKPOINT_DIR / "last_action_diffusion_50k_50ep.pt")
    pd.DataFrame(history).to_csv(WORK / "action_diffusion_50k_50ep_history.csv", index=False)
    
    if train_loss < best_loss:
        best_loss = train_loss
        torch.save(payload, CHECKPOINT_DIR / "best_action_diffusion_50k_50ep.pt")

total_min = (time.time() - start_time) / 60.0
print(f"\n🎉 50-Epoch 50K Action Diffusion Completed in {total_min:.2f} minutes! Best Loss: {best_loss:.5f}")

In [ ]:
# Cell 6: Generate Multi-Action Verification Grid
# Takes 1 Single Character Reference and generates 4 TRUE DISTINCT ACTIONS (Spellcast, Thrust, Walk, Slash)
model.eval()
batch = next(iter(train_loader))
test_char = batch["reference"][0:1].to(DEVICE)

action_titles = ["Action 0: Spellcast", "Action 1: Thrust Attack", "Action 2: Walk Forward", "Action 3: Slash Strike"]
generated_actions = []

for act_id in range(4):
    act_tensor = torch.tensor([act_id], device=DEVICE, dtype=torch.long)
    t_start = 120
    n = torch.randn_like(test_char)
    a_b = alphas_cumprod[t_start]
    cur_x = torch.sqrt(a_b) * test_char + torch.sqrt(1.0 - a_b) * n
    
    with torch.no_grad():
        for step in reversed(range(t_start + 1)):
            t_t = torch.full((1,), step, device=DEVICE, dtype=torch.long)
            p_eps = model(cur_x, t_t, test_char, act_tensor)
            b_s = betas[step]
            a_s = alphas[step]
            m_s = (1.0 / torch.sqrt(a_s)) * (cur_x - (b_s / torch.sqrt(1.0 - alphas_cumprod[step])) * p_eps)
            if step > 0:
                cur_x = m_s + torch.sqrt(b_s) * torch.randn_like(cur_x) * 0.15
            else:
                cur_x = m_s
                
    frame_np = ((cur_x.clamp(-1, 1).cpu().permute(0, 2, 3, 1).numpy()[0] + 1.0) / 2.0).clip(0, 1)
    generated_actions.append(frame_np)

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), dpi=300)
ref_np = ((test_char.cpu().permute(0, 2, 3, 1).numpy()[0] + 1.0) / 2.0).clip(0, 1)
axes[0].imshow(ref_np)
axes[0].axis("off")
axes[0].set_title("Input Character (Idle)", fontsize=11, fontweight="bold")

for i in range(4):
    axes[i+1].imshow(generated_actions[i])
    axes[i+1].axis("off")
    axes[i+1].set_title(action_titles[i], fontsize=11, fontweight="bold", color="#2563eb")

plt.suptitle("GameForge AI: Conditioned Multi-Action Generation from Single Character", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(WORK / "verified_action_50k_verification_grid.png", dpi=300)
plt.show()
print("Saved verified action grid!")

In [ ]:
# Cell 7: Package Final Trained Artifacts into Zip for Download
!zip -r gameforge_verified_action_diffusion_50k_artifacts.zip \
    checkpoints/best_action_diffusion_50k_50ep.pt \
    checkpoints/last_action_diffusion_50k_50ep.pt \
    action_diffusion_50k_50ep_history.csv \
    verified_action_50k_verification_grid.png

print("\n📦 Ready for download: gameforge_verified_action_diffusion_50k_artifacts.zip")